# ⚡ Gated Recurrent Unit (GRU) — The Complete Deep-Dive

**Topics covered:**
1. Motivation — why we need a simpler alternative to LSTM
2. GRU architecture — how it merges LSTM's ideas
3. All GRU equations with full intuition
4. Step-by-step numerical trace through GRU cell
5. ManualGRUCell implementation in PyTorch
6. Bidirectional GRU
7. LSTM vs GRU — side-by-side comparison (math, params, speed, use cases)
8. End-to-end training example with visualisations
9. When to use GRU vs LSTM vs Transformer
10. Senior-level interview Q&A


## 1. 🔥 Why GRU Exists — Simplifying LSTM

LSTM was introduced in 1997 (Hochreiter & Schmidhuber).  
By 2014, **Cho et al. (2014)** asked a key question:

> "LSTM has 4 learnable matrices per step ($W_f, W_i, W_o, W_c$). Do we really need all of them?"

### LSTM's Redundancy
- The **forget gate** decides what to erase from the past.  
- The **input gate** decides how much new info to write.  
- These two decisions are **complementary** — if you erase 70%, you effectively add 30%.  

GRU merges these two gates into a single **Update Gate ($z_t$)**:
$$z_t = 1 - f_t \approx i_t$$
> "What fraction do I *keep* from the past?" = what fraction do I *not write new*."

### What GRU Removes from LSTM
| LSTM Component | GRU Equivalent |
|---|---|
| Cell State ($C_t$) | **Removed** — GRU only has $h_t$ |
| Forget Gate ($f_t$) | Merged into Update Gate ($z_t$) |
| Input Gate ($i_t$) | Merged into Update Gate ($1 - z_t$) |
| Output Gate ($o_t$) | **Removed** — full $h_t$ is always output |
| Candidate ($\tilde{C}_t$) | Kept as Candidate $\tilde{h}_t$ |

**Result:** GRU has **2 gates** instead of **3**, and **1 state** instead of **2** → ~33% fewer parameters and faster training.


## 2. 🏗️ GRU Architecture — Full Equations

At each time step $t$, GRU receives:
- $x_t$ → current input vector
- $h_{t-1}$ → previous hidden state (the only state — no separate cell state)

### Equation 1 — Update Gate ($z_t$): "How much of the past to keep?"
$$z_t = \sigma(W_z \cdot [h_{t-1},\, x_t] + b_z)$$

- Range: $(0, 1)$  
- Close to **1** → keep mostly old hidden state (remember the past)
- Close to **0** → accept mostly new candidate (focus on the present)

---

### Equation 2 — Reset Gate ($r_t$): "How much of the past to expose for the candidate?"
$$r_t = \sigma(W_r \cdot [h_{t-1},\, x_t] + b_r)$$

- Range: $(0, 1)$  
- Close to **0** → ignore $h_{t-1}$ when computing candidate (hard reset)
- Close to **1** → use all of $h_{t-1}$ when computing candidate

> **Key insight:** The reset gate controls how much prior context influences the *candidate state*. A low reset gate forces the candidate to be computed almost from scratch using only the current input.

---

### Equation 3 — Candidate Hidden State ($\tilde{h}_t$): "What would a fresh read give us?"
$$\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1},\, x_t] + b_h)$$

Note the **pointwise multiplication** $r_t \odot h_{t-1}$:  
- If $r_t \approx 0$: candidate ignores past ($[0, x_t]$ effectively)  
- If $r_t \approx 1$: candidate uses full past context ($[h_{t-1}, x_t]$)

---

### Equation 4 — Final Hidden State ($h_t$): "Blend old and new"
$$h_t = z_t \odot h_{t-1} + (1 - z_t) \odot \tilde{h}_t$$

This is the **interpolation step**:
- $z_t = 1$ → $h_t = h_{t-1}$ (copy past hidden state exactly — skip this word)
- $z_t = 0$ → $h_t = \tilde{h}_t$ (replace entirely with fresh candidate)
- $z_t = 0.5$ → blend 50/50

### The Elegant Symmetry
$$h_t = z_t \odot h_{t-1} + (\underbrace{1 - z_t}_{\text{forget}}) \odot \tilde{h}_t$$

The coefficient $(1 - z_t)$ automatically acts as the *forget factor* — no need for a separate forget gate. This is the key design insight.

---

### Parameter Count (GRU vs LSTM)
For input_dim=$D$, hidden_dim=$H$:

| Matrix | LSTM params | GRU params |
|---|---|---|
| Forget (or Update) $W_f / W_z$ | $H(H+D)+H$ | $H(H+D)+H$ |
| Input $W_i$ | $H(H+D)+H$ | — (merged into $z_t$) |
| Candidate $W_c / W_h$ | $H(H+D)+H$ | $H(H+D)+H$ |
| Output $W_o$ | $H(H+D)+H$ | — (removed) |
| Reset $W_r$ | — | $H(H+D)+H$ |
| **Total** | **4 · (H(H+D)+H)** | **3 · (H(H+D)+H)** |

For $D=10, H=128$: LSTM = **70,144** / GRU = **52,608** params → **25% fewer** ✅


In [ ]:
import numpy as np
np.random.seed(4)
np.set_printoptions(precision=4, suppress=True)

def sigmoid(z): return 1 / (1 + np.exp(-z))

# ── Tiny GRU: input_dim=2, hidden_dim=3 ──
D, H = 2, 3

def rand(r, c): return np.random.randn(r, c) * 0.3

Wz = rand(H, H+D); bz = np.zeros(H)   # Update gate
Wr = rand(H, H+D); br = np.zeros(H)   # Reset  gate
Wh = rand(H, H+D); bh = np.zeros(H)   # Candidate

# 3-step sequence
inputs = [np.array([0.9, 0.2]),
          np.array([0.3, 0.8]),
          np.array([0.1, 0.5])]

h = np.zeros(H)   # h_0

print("=" * 65)
print("STEP-BY-STEP GRU FORWARD PASS  (hidden_dim=3, input_dim=2)")
print("=" * 65)

for t, x in enumerate(inputs, 1):
    combined   = np.concatenate([h, x])

    z_t        = sigmoid(Wz @ combined + bz)
    r_t        = sigmoid(Wr @ combined + br)

    # Reset: expose only r * h to candidate
    reset_h    = r_t * h
    cand_input = np.concatenate([reset_h, x])
    h_tilde    = np.tanh(Wh @ cand_input + bh)

    h_new      = z_t * h + (1 - z_t) * h_tilde

    print(f"\n── Step {t} ───────────────────────────────────────────────")
    print(f"  x_{t}             = {x}")
    print(f"  h_{t-1}           = {h.round(4)}")
    print(f"  update gate z_{t}  = {z_t.round(4)}  ← keep {z_t.mean():.0%} of past")
    print(f"  reset  gate r_{t}  = {r_t.round(4)}  ← expose {r_t.mean():.0%} of past to candidate")
    print(f"  r_{t}⊙h_{t-1}       = {reset_h.round(4)}")
    print(f"  candidate h̃_{t}    = {h_tilde.round(4)}")
    print(f"  h_{t} = z⊙h_prev + (1-z)⊙h̃ = {h_new.round(4)}")

    h = h_new

print("\n" + "=" * 65)
print(f"FINAL hidden state h_3 = {h.round(4)}")
print("(This is both the 'memory vault' AND the output — no separate cell state)")
print("=" * 65)


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(7)

class ManualGRUCell(nn.Module):
    """Fully transparent GRU cell — exposes all gate activations."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # Update gate
        self.W_z = nn.Linear(input_dim + hidden_dim, hidden_dim)
        # Reset gate
        self.W_r = nn.Linear(input_dim + hidden_dim, hidden_dim)
        # Candidate
        self.W_h = nn.Linear(input_dim + hidden_dim, hidden_dim)

    def forward(self, x, h_prev):
        combined = torch.cat([h_prev, x], dim=1)

        z = torch.sigmoid(self.W_z(combined))     # update gate
        r = torch.sigmoid(self.W_r(combined))     # reset gate

        # Candidate: gate the past with reset first
        reset_combined = torch.cat([r * h_prev, x], dim=1)
        h_tilde = torch.tanh(self.W_h(reset_combined))

        h_next = z * h_prev + (1 - z) * h_tilde
        return h_next, {'z': z, 'r': r, 'h_tilde': h_tilde}

# ── Run for T steps and record gate activations ──
cell = ManualGRUCell(input_dim=8, hidden_dim=20)
h = torch.zeros(1, 20)
gate_log = {'z': [], 'r': []}

for t in range(15):
    x_t = torch.randn(1, 8)
    h, gates = cell(x_t, h)
    gate_log['z'].append(gates['z'].mean().item())
    gate_log['r'].append(gates['r'].mean().item())

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("ManualGRUCell — Gate Behaviour", color='white', fontsize=13)

ax = axes[0]
ax.set_facecolor('#13131f')
ax.plot(gate_log['z'], 'o-', color='#4ecdc4', lw=2, label='Update gate z_t (keep past)')
ax.plot(gate_log['r'], 's-', color='#fd79a8', lw=2, label='Reset gate r_t (expose past)')
ax.set_title("Update vs Reset Gate Activations", color='white')
ax.set_xlabel('Time step', color='#aaa')
ax.set_ylabel('Mean activation', color='#aaa')
ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
ax.set_ylim(0, 1)
for sp in ax.spines.values(): sp.set_color('#333')

ax = axes[1]
ax.set_facecolor('#13131f')
# Semantic example: Update gate near 1 → skip the word (function words)
words = ['The','quick','brown','fox','jumps','over','the','lazy','dog','!']
z_sim = [0.9, 0.3, 0.2, 0.15, 0.2, 0.85, 0.9, 0.85, 0.2, 0.5]
r_sim = [0.4, 0.9, 0.95, 0.9, 0.85, 0.3, 0.4, 0.3, 0.9, 0.6]
x = range(len(words))
ax.bar(x, z_sim, label='Update z (1 = keep past)', color='#4ecdc4', alpha=0.7)
ax.bar(x, r_sim, label='Reset r (1 = use past in candidate)', color='#fd79a8',
       alpha=0.6, bottom=0)
ax.set_xticks(list(x))
ax.set_xticklabels(words, rotation=30, color='#ccc', fontsize=9)
ax.set_title('Simulated Gate Pattern — "The quick brown fox…"', color='white')
ax.set_ylabel('Gate value', color='#aaa')
ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')

plt.tight_layout()
plt.savefig('notes/assets/gru_gates.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("ManualGRUCell works correctly ✅")


## 3. ⚖️ GRU vs LSTM — Full Comparison

### Architecture Diagram (Text)
```
LSTM (per cell):                  GRU (per cell):
┌──────────────────────────┐      ┌─────────────────────────┐
│  [h_{t-1}, x_t]          │      │  [h_{t-1}, x_t]         │
│     ↓  ↓  ↓  ↓          │      │       ↓       ↓         │
│   Wf  Wi  Wc  Wo         │      │      Wz      Wr         │
│  σ()  σ() tanh σ()      │      │     σ()     σ()         │
│   f    i   C̃   o         │      │      z       r          │
│   ↓    ↓   ↓             │      │      ↓       ↓         │
│  C_{t-1}×f + i×C̃ = C_t  │      │   r×h+x → Wh → tanh(·)  │
│  h_t = o ⊙ tanh(C_t)    │      │   h_t = z×h + (1-z)×h̃  │
└──────────────────────────┘      └─────────────────────────┘
    2 states, 4 gates                 1 state, 2 gates
```

---

### Detailed Comparison Table

| Dimension | LSTM | GRU |
|---|---|---|
| **States** | 2: $h_t$ (hidden) + $C_t$ (cell) | 1: $h_t$ only |
| **Gates** | 3: Forget, Input, Output | 2: Update, Reset |
| **Equations** | 6 equations | 4 equations |
| **Params (H=128, D=64)** | ~200K | ~150K (~25% fewer) |
| **Training speed** | Slower | ~25-33% faster |
| **Long-range memory** | Excellent (Cell State highway) | Good (Update gate can hold) |
| **Short-range tasks** | Over-engineered | Ideal |
| **Vanishing gradient** | Solved via CEC | Solved via additive interpolation |
| **Expressiveness** | Slightly higher | Slightly lower |
| **Standard on NLP** | More common (pre-Transformer) | Competitive, especially for smaller datasets |

---

### When to Use Which

**Prefer LSTM when:**
- Very long sequences (>100 steps) where fine-grained memory control is critical
- Tasks with complex, nested temporal dependencies (e.g., multi-clause parsing)
- You have abundant compute and training time
- The problem is linguistically rich (e.g., long-form text generation)

**Prefer GRU when:**
- Smaller dataset (fewer params → less overfitting)
- Real-time latency is important (faster inference)
- You want to tune quickly in a research/prototype phase
- Sequences are short-to-medium length (<100 steps)
- Speech processing, smaller NLP tasks

**Prefer Transformer (neither) when:**
- Sequences are very long (500+ tokens)
- You have a large GPU cluster enabling full parallelism
- The task benefits from bidirectional global attention (BERT-style)
- You're building a production LLM or large-scale system


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("GRU vs LSTM — Side-by-Side Comparison", color='white', fontsize=14)

# Data
categories = ['Parameters\n(H=128, D=64)', 'Training\nSpeed (rel.)',
               'Long-range\nMemory', 'Short-task\nFit', 'Vanishing\nGrad Resistance']
lstm_scores = [70, 60, 95, 65, 90]
gru_scores  = [52, 85, 80, 90, 82]

x = np.arange(len(categories))
w = 0.35

# ── Bar chart ──
ax = axes[0]
ax.set_facecolor('#13131f')
b1 = ax.bar(x - w/2, lstm_scores, w, label='LSTM', color='#a29bfe', alpha=0.85)
b2 = ax.bar(x + w/2, gru_scores,  w, label='GRU',  color='#4ecdc4', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=9, color='#ccc')
ax.set_title('LSTM vs GRU — Capability Scores', color='white')
ax.set_ylabel('Score (higher = better)', color='#aaa')
ax.legend(fontsize=10); ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')
for bar in b1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                       str(bar.get_height()), ha='center', color='#a29bfe', fontsize=8)
for bar in b2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                       str(bar.get_height()), ha='center', color='#4ecdc4', fontsize=8)

# ── Gradient flow comparison ──
ax = axes[1]
ax.set_facecolor('#13131f')
steps = np.arange(60)
rnn_grad  = (0.63)**steps
gru_grad  = (0.92)**steps   # update gate ≈ 0.92
lstm_grad = (0.95)**steps   # forget gate ≈ 0.95

ax.semilogy(steps, rnn_grad,  color='#ff6b6b', lw=2, label='RNN (baseline)')
ax.semilogy(steps, gru_grad,  color='#4ecdc4', lw=2, label='GRU (z≈0.92)')
ax.semilogy(steps, lstm_grad, color='#a29bfe', lw=2, label='LSTM (f≈0.95)')
ax.axhline(1e-3, color='white', ls=':', lw=1.0, alpha=0.7, label='~0.001 threshold')
ax.set_xlabel('Steps back', color='#aaa')
ax.set_ylabel('Gradient (log scale)', color='#aaa')
ax.set_title('Gradient Flow: RNN vs GRU vs LSTM', color='white')
ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')

# ── Radar/Spider chart ──
ax = axes[2]
ax.set_facecolor('#13131f')
dims = ['Long-range
Memory', 'Training
Speed', 'Low
Params',
        'Short-task
Fit', 'Gradient
Resistance']
N = len(dims)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

def to_radar(vals): return vals + vals[:1]

lstm_r = [0.95, 0.60, 0.50, 0.65, 0.90]
gru_r  = [0.80, 0.85, 0.80, 0.90, 0.82]

ax2 = plt.subplot(1, 3, 3, polar=True)
ax2.set_facecolor('#13131f')
ax2.plot(angles, to_radar(lstm_r), 'o-', color='#a29bfe', lw=2, label='LSTM')
ax2.fill(angles, to_radar(lstm_r), alpha=0.15, color='#a29bfe')
ax2.plot(angles, to_radar(gru_r),  's-', color='#4ecdc4', lw=2, label='GRU')
ax2.fill(angles, to_radar(gru_r),  alpha=0.15, color='#4ecdc4')
ax2.set_xticks(angles[:-1]); ax2.set_xticklabels(dims, fontsize=9, color='#ccc')
ax2.set_ylim(0, 1); ax2.set_facecolor('#13131f')
ax2.tick_params(colors='#666')
ax2.set_title('Capability Radar', color='white', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
for line in ax2.xaxis.get_gridlines(): line.set_color('#333')
for line in ax2.yaxis.get_gridlines(): line.set_color('#333')

plt.tight_layout()
plt.savefig('notes/assets/gru_vs_lstm.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Saved → notes/assets/gru_vs_lstm.png")


## 4. 🛠️ End-to-End Training with Bidirectional GRU

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(55)

VOCAB, SEQ, BATCH = 3000, 20, 16

def make_data(n, pos=True):
    X = torch.randint(1, VOCAB, (n, SEQ))
    X[:, :4] = torch.randint(1500, 2200, (n, 4)) if pos else torch.randint(100, 600, (n, 4))
    y = torch.ones(n, dtype=torch.long) if pos else torch.zeros(n, dtype=torch.long)
    return X, y

Xp, yp = make_data(200, True);  Xn, yn = make_data(200, False)
X_all   = torch.cat([Xp,Xn]);   y_all  = torch.cat([yp,yn])
perm    = torch.randperm(400);  X_all, y_all = X_all[perm], y_all[perm]
X_tr, y_tr = X_all[:300], y_all[:300]
X_va, y_va = X_all[300:],  y_all[300:]

class BiGRU_Classifier(nn.Module):
    """Bidirectional 2-layer GRU for binary classification."""
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB, 32, padding_idx=0)
        self.gru  = nn.GRU(32, 64, num_layers=2, batch_first=True,
                           bidirectional=True, dropout=0.3)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(128, 2)   # 64*2 for BiGRU

    def forward(self, x):
        e = self.drop(self.emb(x))
        _, h = self.gru(e)              # h: (num_layers*2, batch, hidden)
        ctx = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(ctx))

model   = BiGRU_Classifier()
opt     = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
loss_fn = nn.CrossEntropyLoss()

tr_losses, va_accs = [], []

for epoch in range(60):
    model.train(); ep_loss = 0
    perm = torch.randperm(300)
    for b in range(0, 300, BATCH):
        idx = perm[b:b+BATCH]; xb, yb = X_tr[idx], y_tr[idx]
        logits = model(xb); loss = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); ep_loss += loss.item()
    sched.step(); tr_losses.append(ep_loss / (300 // BATCH))
    model.eval()
    with torch.no_grad():
        va_accs.append((model(X_va).argmax(1) == y_va).float().mean().item())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes: ax.set_facecolor('#13131f')

axes[0].plot(tr_losses, color='#4ecdc4', lw=2)
axes[0].fill_between(range(len(tr_losses)), tr_losses, alpha=0.15, color='#4ecdc4')
axes[0].set_title('Training Loss', color='white'); axes[0].set_xlabel('Epoch', color='#aaa')
axes[0].tick_params(colors='#aaa')

axes[1].plot(va_accs, color='#fd79a8', lw=2)
axes[1].fill_between(range(len(va_accs)), va_accs, alpha=0.15, color='#fd79a8')
axes[1].axhline(0.5, color='#fdcb6e', ls='--', lw=1, label='Baseline')
axes[1].set_title(f'Val Accuracy (Final: {va_accs[-1]:.1%})', color='white')
axes[1].set_xlabel('Epoch', color='#aaa'); axes[1].legend()
axes[1].tick_params(colors='#aaa')

for ax in axes:
    for sp in ax.spines.values(): sp.set_color('#333')

plt.suptitle("Bidirectional GRU — Sentiment Training", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('notes/assets/gru_training.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

# Params comparison
lstm_params = sum(p.numel() for p in nn.LSTM(32, 64, 2, bidirectional=True).parameters())
gru_params  = sum(p.numel() for p in nn.GRU(32, 64, 2, bidirectional=True).parameters())
print(f"BiGRU params:  {gru_params:,}")
print(f"BiLSTM params: {lstm_params:,}")
print(f"GRU savings:  {(1 - gru_params/lstm_params):.1%} fewer parameters")
print(f"Final val accuracy: {va_accs[-1]:.1%}")


## 5. 📋 GRU Equations — Quick Reference Card

| Symbol | Name | Equation | Range |
|---|---|---|---|
| $z_t$ | Update gate | $\sigma(W_z[h_{t-1},x_t]+b_z)$ | $(0,1)$ |
| $r_t$ | Reset gate | $\sigma(W_r[h_{t-1},x_t]+b_r)$ | $(0,1)$ |
| $\tilde{h}_t$ | Candidate | $\tanh(W_h[r_t \odot h_{t-1},x_t]+b_h)$ | $(-1,1)$ |
| $h_t$ | Hidden state | $z_t \odot h_{t-1} + (1-z_t) \odot \tilde{h}_t$ | $(-1,1)$ |

---

## 6. 🎯 Senior-Level Interview Q&A

**Q1: How does GRU avoid vanishing gradients?**  
> The final hidden state equation is:  
> $h_t = z_t \odot h_{t-1} + (1-z_t) \odot \tilde{h}_t$  
> Taking the gradient:  $\partial h_t / \partial h_{t-1} = z_t$  
> So the gradient chain $\partial h_T / \partial h_k = \prod_{i=k+1}^T z_i$.  
> If the network learns $z_i \approx 1$ for a stretch, this product stays near 1 — no vanishing. Exactly the same principle as LSTM's forget gate.

---

**Q2: Why does GRU have a reset gate if it already has an update gate?**  
> The two gates operate at **different points in the computation**:  
> - The **reset gate** controls how much of $h_{t-1}$ is used when *computing the candidate* $\tilde{h}_t$  
> - The **update gate** controls how much of $h_{t-1}$ vs $\tilde{h}_t$ is used in the *final output* $h_t$  
> Without the reset gate, the candidate $\tilde{h}_t$ would always be strongly influenced by the full $h_{t-1}$, even when the network wants to write fresh information. The reset gate allows the candidate to be computed almost from scratch (hard reset).

---

**Q3: In practice, which performs better — GRU or LSTM?**  
> Empirically, performance is task-dependent and the difference is often small. Key findings (Chung et al. 2014, Jozefowicz et al. 2015):  
> - GRU tends to converge faster with less data  
> - LSTM tends to perform better on very long sequences  
> - On most NLP benchmarks they're statistically tied  
> The practical recommendation: start with GRU for fast prototyping; if the task requires long-range dependencies (>100 steps), upgrade to LSTM.

---

**Q4: Can you have a reset gate $pprox 0$ and update gate $pprox 0$ simultaneously?**  
> Yes. In this case:  
> - $r_t \approx 0$ → candidate $\tilde{h}_t \approx \tanh(W_h [0, x_t])$ — reads only current input  
> - $z_t \approx 0$ → $h_t \approx \tilde{h}_t$ — completely replaces hidden state with candidate  
> This is a "full reset" — the model completely forgets the past and re-reads only the current input. Useful at sentence boundaries.

---

**Q5: What hyperparameter changes matter most for GRU?**
> 1. **Hidden size**: Biggest lever for capacity. Try 64 → 128 → 256  
> 2. **Num layers**: 1 layer often enough for short sequences; 2-3 for complex tasks  
> 3. **Dropout**: Apply between stacked layers, not within the recurrent step  
> 4. **Forget gate bias trick (GRU equivalent)**: Init $b_z$ to a positive value (e.g., 1.0) to encourage remembering by default in early training  
> 5. **Gradient clipping**: Always use threshold 1–5 with RNN/GRU/LSTM

---

**Q6: How is Bidirectional GRU different from a regular GRU?**  
> Two GRU layers run in opposite directions over the same input:  
> - Forward GRU: $h_t^{\rightarrow}$ — has seen words 1 to t  
> - Backward GRU: $h_t^{\leftarrow}$ — has seen words T down to t  
> Final representation at position t: $H_t = [h_t^{\rightarrow}; h_t^{\leftarrow}]$ (concatenated)  
> This doubles the hidden dimension and the parameter count roughly by 2×.  
> Cannot be used for autoregressive/causal generation — use only when the full sequence is available at inference time.
